In [22]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")
from utils.scaling import fit_scalers, apply_scaler
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75anno_articulatory_features

In [23]:
df = pd.read_csv("../dataset_construction/datasets/processed_aemri75anno.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])

In [24]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.85,1.87,1.87,2.04,ð,ɛ,ð,then,0,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,1.87,2.04,2.04,2.07,ɛ,n,ð_ɛ,then,0,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,2.07,2.09,2.09,2.14,ð,ʌ,ð,the,1,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,2.14,2.18,2.18,2.24,n,ɔ,n,north,2,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,2.18,2.24,2.24,2.39,ɔ,ɹ,n_ɔ,north,2,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...,...,...,...,...,...,...
26851,52.92,52.97,52.97,53.01,æ,ŋ,l_æ,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26852,52.97,53.01,53.01,53.04,ŋ,g,l_æ_ŋ,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26853,53.01,53.04,53.04,53.10,g,w,l_æ_ŋ_g,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26854,53.04,53.10,53.10,53.28,w,ɨ,l_æ_ŋ_g_w,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(174, 2),
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
).to(device)
ckpt = torch.load(
    "../../encoder_training/ckpts/aemri75anno/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,        # was 1024. With ~30k tokens that was memorisation
                          # territory, and Euclidean distances concentrate badly
                          # in 1024-d, which is bad for what you use z for.
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/aemri75anno/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [26]:
with open(f"../../encoder_training/scalers/aemri75anno_art.pkl", "rb") as f:
    art_scalers = pickle.load(f)
with open(f"../../encoder_training/scalers/aemri75anno_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [27]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,1.85,1.87,1.87,2.04,ð,ɛ,ð,then,0,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
1,1.87,2.04,2.04,2.07,ɛ,n,ð_ɛ,then,0,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
2,2.07,2.09,2.09,2.14,ð,ʌ,ð,the,1,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
3,2.14,2.18,2.18,2.24,n,ɔ,n,north,2,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
4,2.18,2.24,2.24,2.39,ɔ,ɹ,n_ɔ,north,2,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub043/2drt/video/su...,sub043
...,...,...,...,...,...,...,...,...,...,...,...,...
26851,52.92,52.97,52.97,53.01,æ,ŋ,l_æ,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26852,52.97,53.01,53.01,53.04,ŋ,g,l_æ_ŋ,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26853,53.01,53.04,53.04,53.10,g,w,l_æ_ŋ_g,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071
26854,53.04,53.10,53.10,53.28,w,ɨ,l_æ_ŋ_g_w,language,134,/Volumes/Transcend/USC-75/USC-75SpeakerAnnot16...,/Volumes/Transcend/USC-75/sub071/2drt/video/su...,sub071


In [28]:
def get_vectors(art_f, ac_f, start, end, subject):
    articulatory_features = get_aemri75anno_articulatory_features(
        art_f,
        start,
        end
    )

    if articulatory_features is not None:
        articulatory_features = articulatory_features.reshape(
            articulatory_features.shape[0],
            174,
            2,
        )
        articulatory_features = apply_scaler(articulatory_features, art_scalers[subject])

        articulatory_features = (
            torch.from_numpy(articulatory_features)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        art_out, art_z = art_model(
            articulatory_features,
            return_latent=True
        )
    else:
        art_out, art_z = None, None

    mfccs = get_mfccs(ac_f, start, end)

    if mfccs is not None:
        mfccs = apply_scaler(mfccs, ac_scalers[subject])
        mfccs = (
            torch.from_numpy(mfccs)
            .float()
            .unsqueeze(0)
            .to(device)
        )

        ac_out, ac_z = ac_model(mfccs, return_latent=True)
    else:
        ac_out, ac_z = None, None

    return {
        "articulatory_features": (
            articulatory_features.detach().cpu().numpy()
            if articulatory_features is not None
            else None
        ),
        "art_z": (
            art_z.detach().cpu().numpy()
            if art_z is not None
            else None
        ),
        "mfccs": (
            mfccs.detach().cpu().numpy()
            if mfccs is not None
            else None
        ),
        "ac_z": (
            ac_z.detach().cpu().numpy()
            if ac_z is not None
            else None
        ),
    }

In [29]:
def distance(v1, v2):
    if v1 is None or v2 is None:
        return None
    return np.linalg.norm(v1 - v2)

In [30]:
pbar = tqdm.tqdm(df.iterrows(), total=len(df))

dists = {
    "art_latent_dist": [],
    # "art_raw_dist": [],
    "ac_latent_dist": [],
    # "ac_raw_dist": [],
}
for idx, data in pbar:
    
    phone1_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone1_start"], data["phone1_end"], data["subject"])
    phone2_vectors = get_vectors(data["art_f"], data["ac_f"], data["phone2_start"], data["phone2_end"], data["subject"])

    art_latent_dist = distance(phone1_vectors["art_z"], phone2_vectors["art_z"])
    # art_raw_dist = distance(phone1_vectors["articulatory_features"], phone2_vectors["articulatory_features"])

    ac_latent_dist = distance(phone1_vectors["ac_z"], phone2_vectors["ac_z"])
    # ac_raw_dist = distance(phone1_vectors["mfccs"], phone2_vectors["mfccs"])

    dists["art_latent_dist"].append(art_latent_dist)
    # dists["art_raw_dist"].append(art_raw_dist)
    dists["ac_latent_dist"].append(ac_latent_dist)
    # dists["ac_raw_dist"].append(ac_raw_dist)

pd.DataFrame(dists).to_csv("distances_aemri75anno.csv", index=False)

100%|██████████| 26856/26856 [2:38:20<00:00,  2.83it/s]  
